# Declaration of Originality

**School of Informatics & IT**
<br/>**Diploma in Applied Artificial Intelligence**
<br/>**Machine Learning for Developers (CAI2C08)**
<br/>**AY2026/2027 April Semester**
<br/>**Program Codes**

* Student Name: _TODO — exact name as it appears in TP-LMS_
* Student ID: 2503051A


**Declaration of Originality**
* I am the originator of this work, and I have appropriately acknowledged all other original sources used as my references for this work.
* I understand that Plagiarism is the act of taking and using the whole or any part of another person’s work, including work generated by AI, and presenting it as my own.
* I understand that Plagiarism is an academic offence and if I am found to have committed or abetted the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.

**Use of Generative AI**

Claude Code was used for project scaffolding, code review and structuring. All modelling decisions, metric justification and analysis are my own and I can explain each of them. _(Declaring this does not affect marks — see spec, Guidelines on using Gen AI.)_


# Libraries

In [ ]:
## Import libraries
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.io import arff

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
)

# Fixed seed everywhere: the rubric's tuning row requires a reproducible setup.
RANDOM_STATE = 42

ARTIFACTS = Path('artifacts')
ARTIFACTS.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)


# 1. Business Understanding

**Goal:** flag phishing websites from 30 structural signals extracted from a URL and its page, so a browser extension / email gateway can warn a user *before* they enter credentials.

**Task type:** binary classification. Target `Result` — `-1` phishing, `1` legitimate.

**Cost of the two errors (this drives the metric choice in section 5):**

| Error | What happens | Who pays |
| --- | --- | --- |
| False negative — phishing scored legitimate | user is not warned, enters credentials | account takeover, financial loss |
| False positive — legitimate scored phishing | user sees a warning on a safe site | one dismissed banner |

These costs are **not symmetric**, and that asymmetry — not the class ratio — is why accuracy is the wrong headline metric here. The classes are close to balanced (confirmed in 2.3.1.1), so accuracy is not *broken*; it is simply blind to which kind of mistake was made.

> **TODO (my own words):** expand the paragraph above into your own framing of who the customer is and what a wrong prediction costs them. This is the paragraph section 5 points back to, and the one the investor Q&A will push on.

**Assumptions**

> **TODO (my own words):** state them, e.g. the 30 features can be recomputed cheaply at request time; phishing tactics in the training data still resemble today's; a warning is advisory, not a hard block.


# 2. Data Understanding

## 2.1 Load dataset

Source: UCI Phishing Websites, id=327 (https://archive.ics.uci.edu/dataset/327/phishing+websites). Tutor-approved at Checkpoint 1. 11,055 rows x 30 features + target.

Loaded from `Training Dataset.arff`. **Not** `.old.arff` — that is a different, 2,456-row file from the same download that encodes binary features as {0,1} instead of {-1,1}, so it would silently change what every feature means.


In [ ]:
## Read the ARFF file into a pandas DataFrame
raw, meta = arff.loadarff('Training Dataset.arff')
df = pd.DataFrame(raw)

# scipy returns ARFF nominal values as bytes (b'-1'); convert to signed ints.
df = df.apply(lambda s: s.str.decode('utf-8')).astype(int)

TARGET = 'Result'
FEATURES = [c for c in df.columns if c != TARGET]

print(f'shape: {df.shape}  |  {len(FEATURES)} features')
df.head()


## 2.2 Summary Statistics

In [ ]:
## Understand the type of variable for each column
df.info()


In [ ]:
## Check for missing data
print('missing values:', df.isna().sum().sum())
print('duplicate rows:', df.duplicated().sum())


In [ ]:
## Describe data distribution
# Every column is a nominal code, so value counts are more informative than
# mean/std — a 'mean' of -0.3 on a {-1,1} column has no meaning.
cardinality = df[FEATURES].nunique().value_counts().rename('n_features')
print(cardinality.rename_axis('n_distinct_values'))

df.describe().T


> **TODO (my own words):** note what the cardinality split tells you: 22 features are binary {-1,1} and 8 are ternary {-1,0,1}. Say why this rules out histograms/scaling and points to crosstabs instead.


## 2.3 Data Visualization

### 2.3.1 Understanding distribution of data

### 2.3.1.1 Understanding distribution of target

In [ ]:
## Understanding distribution of target
counts = df[TARGET].value_counts().sort_index()
labels = {-1: 'Phishing', 1: 'Legitimate'}

ax = sns.barplot(x=[labels[i] for i in counts.index], y=counts.values)
ax.set(title='Class balance of Result', xlabel='', ylabel='Websites')
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v:,} ({v / len(df):.1%})', ha='center', va='bottom')
plt.show()

ratio = counts.max() / counts.min()
print(f'majority:minority = {ratio:.2f}:1')


> **TODO (my own words):** interpret. The ratio is ~1.26:1 — near balanced. Be explicit that this means you are **not** choosing recall because of imbalance; you are choosing it because of the cost asymmetry in section 1. Also note what a majority-class baseline would therefore score (~56%), which is the number section 4 has to beat.


### 2.3.1.2 Understanding distribution of features

In [ ]:
## Understanding distribution of features
## TODO: crosstab / stacked bar of each feature against Result.
## Categorical -> proportion of phishing within each feature level.


> **TODO (my own words):** for each of the features that stands out, write 2-3 sentences of *security* interpretation — why an absent SSL certificate or a very young domain would correlate with phishing, and what that implies about which features the model should lean on. The rubric's A band for EDA (8 marks) is won here, not by the plots themselves. `Phishing Websites Features.docx` defines what each coded level means — use it.


### 2.3.2 Understanding relationship between variables

In [ ]:
## Understanding relationship between variables
## TODO: correlation heatmap over the encoded feature set.


> **TODO (my own words):** call out any strongly correlated pairs and what you will do about them.


# 3. Data Preparation

## 3.1 Data Cleaning

The 30 features were extracted from raw URLs and page HTML by the original researchers and are already encoded as nominal codes. There is no raw URL string in this dataset, so there is nothing to engineer *from*.

**This project therefore uses feature selection, not feature engineering.** The spec asks for "feature engineering and/or feature selection", so selection alone is compliant — inventing transformations of already-engineered codes would be noise dressed up as work. Selection is done in section 5.1.


**Duplicates — the one real data-quality problem in this dataset.**

There are no missing values, but 11,055 rows collapse to only **5,785 unique feature vectors** (47% of rows are exact duplicates of another row). Two consequences:

1. **Leakage.** With a random train/test split, roughly half the test rows have an identical twin sitting in the training set. A tree model can memorise the twin and 'predict' it perfectly, so the reported test scores would be optimistic — they would measure recall on rows the model has effectively already seen.
2. **A hard accuracy ceiling.** 64 feature vectors appear with *both* labels (357 rows). Those are genuinely indistinguishable given these 30 features, so no model can get all of them right. That is irreducible error, not a modelling failure.

This matters more than anything else in cleaning, and it is not obvious from `df.info()` — it only shows up if you check.


In [ ]:
## Quantify the duplication before deciding what to do about it
print(f'rows                      {len(df):,}')
print(f'unique feature vectors    {df[FEATURES].drop_duplicates().shape[0]:,}')
print(f'unique rows incl. label   {df.drop_duplicates().shape[0]:,}')

conflicts = df.groupby(FEATURES)[TARGET].nunique()
print(f'feature vectors with both labels: {(conflicts > 1).sum()}')


> **TODO (my own words):** decide and defend: **drop duplicates** (honest generalisation estimate, but the dataset halves to ~5.8k rows) or **keep them** (treat repeats as real frequency information, but accept that the test score is inflated by leakage). Whichever you pick, say so explicitly and say what it costs you — a lecturer who spots the duplication will ask, and 'I didn't check' is the only wrong answer.

If you keep them, report the de-duplicated score somewhere as a sanity check so the gap between the two is visible.


In [ ]:
## Clean data
## TODO: apply the duplicate decision above (and nothing else — there is no missing
## data and no out-of-range codes to fix).


## 3.2 Train-Test Split

In [ ]:
## Split data into train set and test set
X = df[FEATURES]
y = df[TARGET]

# Stratified so the test set keeps the real class ratio — otherwise recall on the
# phishing class is measured against a distribution that does not exist.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'train {X_train.shape}  test {X_test.shape}')


# 4. Modelling

## 4.1 Baseline

`DummyClassifier(strategy='most_frequent')` always predicts the majority class. Every model below has to beat it, otherwise it has learned nothing. The rubric's A band for model comparison requires test-set results *with* a baseline comparison.


In [ ]:
## Majority-class baseline
## TODO: fit DummyClassifier and record its metrics.


### 4.2 Train Model

Three distinct algorithm families, all on the full 30 features (the 'before' half of the feature-selection comparison in 5.1):

- **LogisticRegression** — linear, coefficients are directly readable
- **RandomForestClassifier** — bagged trees, captures interactions
- **GradientBoostingClassifier** — boosted trees, usually strongest, slowest to train

scikit-learn only, per the spec.


In [ ]:
## Initialise and train models
## TODO: fit the three models, collect train+test metrics into one table.
## Keep the train/test gap — it is the overfitting evidence for the qualitative
## comparison, and it goes to artifacts/baseline_metrics.csv.


> **TODO (my own words):** quantitative comparison (the metrics table) **and** qualitative: interpretability, training time, overfitting risk, what each would cost to serve in the app. Then justify which model you carry into tuning on performance *and* deployment needs — not 'highest accuracy'.


# 5. Model Evaluation

## Evaluation metric selection

> **TODO (my own words):** state which metric you optimise and tie it back to section 1's cost table in a few sentences. Report the full picture regardless — precision, recall, F1 and the confusion matrix — so the trade-off you accepted is visible, not asserted.


In [ ]:
## Evaluate model
## TODO: classification_report + ConfusionMatrixDisplay for the selected model.


## Iterative model development

Order matters: baselines on all 30 features (section 4) come first, so 5.1 has a genuine 'before' to compare against.


### 5.1 Feature selection

Two independent methods, cross-checked against each other:

1. **RandomForest feature importances** — model-based, captures interactions
2. **`SelectKBest(mutual_info_classif)`** — statistical, no model assumed; mutual information suits categorical features where a linear correlation would not

Agreement between two different methods is stronger evidence than either alone.


In [ ]:
## Further feature selection
## TODO: rank by RF importance and by mutual information, compare the two rankings,
## pick the surviving set -> artifacts/selected_features.json


In [ ]:
## Before / after: all 30 features vs reduced set
## TODO: refit the three models on the reduced set -> artifacts/reduced_metrics.csv


> **TODO (my own words):** did performance hold, improve, or trade off? Tie the decision to reduced complexity and to how many inputs the Streamlit user has to fill in — not to a metric number alone. This row is worth 10 marks, the largest single one.


### 5.2 Hyperparameter tuning

`RandomizedSearchCV` only, **exactly 2 hyperparameters, at most 3 values each** — a hard spec constraint. `random_state=RANDOM_STATE` throughout for reproducibility.


In [ ]:
## RandomizedSearchCV
## TODO: log the grid, the full cv_results_ table -> artifacts/search_results.csv,
## and the best params.


In [ ]:
## Tuned vs untuned on the test set
## TODO: side-by-side table, not a sentence. The rubric asks for documented
## improvement vs baseline.


> **TODO (my own words):** explain the impact and why these final settings.


### 5.3 Final model


In [ ]:
## Persist for the Streamlit app
## TODO: joblib.dump the fitted model -> artifacts/final_model.pkl
## and the feature list the app must collect, in the same column order.


In [ ]:
## New data

## Predict
## TODO: sanity-check one hand-built phishing-looking row and one legitimate-looking
## row. This is also the smoke test for the app in S8.
